# **Model Selection**

In [ ]:
# ════════════════════════════════════════════════════════════════
#  ★  MODEL SELECTION  ★
# ════════════════════════════════════════════════════════════════
MODEL_NAME = 'dense_unet'   # Our novel architecture
# ════════════════════════════════════════════════════════════════

print(f"Selected model: {MODEL_NAME}")

# **Dependencies**

In [ ]:
# Install dependencies (run once)
!pip install torch torchvision opencv-python tqdm matplotlib numpy --quiet
# !pip install torch==2.0.1+cu117 torchvision==0.15.2+cu117 --index-url https://download.pytorch.org/whl/cu117 -q

In [ ]:
import os, sys, json, time, random, shutil, math, warnings
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.models import resnet50, ResNet50_Weights

# Mixed-precision training (AMP) — speeds up training on GPU with no accuracy loss
from torch.cuda.amp import GradScaler, autocast

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()          # AMP only on GPU
# USE_AMP=False
print(f'Device    : {DEVICE}')
print(f'PyTorch   : {torch.__version__}')
print(f'CUDA      : {torch.cuda.is_available()}')
print(f'AMP (fp16): {USE_AMP}  — training will be {"faster (mixed precision)" if USE_AMP else "CPU mode"}')
print(f'Model     : {MODEL_NAME}')

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # NOTE: deterministic=True is accurate but slightly slower.
    # Switch benchmark=True / deterministic=False for max speed (minor reproducibility trade-off).
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(42)
print('Seed set to 42')

# **Training Configurations**

In [ ]:
@dataclass
class TrainConfig:
    # ── Paths ──────────────────────────────────────────────────
    tile_image_dir : str   = './tiles/images'
    tile_mask_dir  : str   = './tiles/masks'
    output_dir     : str   = './runs'

    # ── Data split ─────────────────────────────────────────────
    train_ratio    : float = 0.70
    val_ratio      : float = 0.15
    # test  = remainder

    # ── Training ───────────────────────────────────────────────
    epochs              : int   = 6
    batch_size          : int   = 8
    learning_rate       : float = 1e-4
    weight_decay        : float = 1e-5
    early_stop_patience : int   = 10

    # ── DataLoader ─────────────────────────────────────────────
    # num_workers: set to 0 on Windows to avoid multiprocessing issues
    num_workers : int  = 2
    pin_memory  : bool = True

    # ── Loss weights ───────────────────────────────────────────
    bce_weight  : float = 0.5
    dice_weight : float = 0.5
    pos_weight  : float = 10.0    # high weight for sparse rim pixels

    # ── Model ──────────────────────────────────────────────────
    in_channels  : int = 1
    out_channels : int = 1
    seed         : int = 42

    # ── Injected from global MODEL_NAME ────────────────────────
    model_name : str = MODEL_NAME

CFG = TrainConfig()
print(f'Config loaded  →  model_name = {CFG.model_name}')
print(f'   epochs={CFG.epochs}  batch={CFG.batch_size}  lr={CFG.learning_rate}')

In [ ]:
# ── Tiling paths ──────────────────────────────────────────────────────────────
IMAGE_DIR = r"/kaggle/input/datasets/sahibjparmar/mars-crater-dataset-for-semantic-segmentation/Dataset/Images"
MASK_DIR  = r"/kaggle/input/datasets/sahibjparmar/mars-crater-dataset-for-semantic-segmentation/Dataset/Mars_Crater_Segmentation_Dataset_02-32_km_radius/edge_thicker"

OUTPUT_IMAGE_DIR = "./tiles/images"
OUTPUT_MASK_DIR  = "./tiles/masks"

TILE_SIZE             = 512
STRIDE                = 512      # no overlap; set < TILE_SIZE for overlap
ENABLE_COPY_PASTE     = True
PASTES_PER_IMAGE      = 3        # craters to paste per augmented tile
AUGMENTED_COPIES      = 2        # augmented versions per original tile
MIN_CRATER_PIXELS     = 30
BLEND_FEATHER_RADIUS  = 5
MAX_PASTE_ATTEMPTS    = 10
BLACK_STRIP_THRESHOLD = 0.15     # >15% near-black pixels → skip tile
BLACK_PIXEL_VALUE     = 10

os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)
os.makedirs(OUTPUT_MASK_DIR,  exist_ok=True)
print('Output directories ready')

# **Augmentation**

In [ ]:
# ── Artifact Detection ────────────────────────────────────────────────────────
def has_black_strip(image: np.ndarray, threshold=BLACK_STRIP_THRESHOLD) -> bool:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    return (gray < BLACK_PIXEL_VALUE).sum() / gray.size > threshold

# ── Crater Instance Library ───────────────────────────────────────────────────
class CraterLibrary:
    def __init__(self):
        self.craters = []

    def extract_from_tile(self, image: np.ndarray, mask: np.ndarray):
        _, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
        for i in range(1, len(stats)):
            x, y, w, h, area = stats[i]
            if area < MIN_CRATER_PIXELS:
                continue
            pad = max(5, int(max(w, h) * 0.1))
            x1, y1 = max(0, x-pad), max(0, y-pad)
            x2, y2 = min(image.shape[1], x+w+pad), min(image.shape[0], y+h+pad)
            self.craters.append({'img': image[y1:y2, x1:x2].copy(),
                                 'mask': mask[y1:y2, x1:x2].copy(),
                                 'radius': int(max(w, h) / 2), 'area': area})

    def sample(self, prefer_small=True):
        if not self.craters:
            return None
        if prefer_small and random.random() < 0.65:
            radii = np.array([c['radius'] for c in self.craters], dtype=float)
            weights = 1.0 / (radii + 1e-6); weights /= weights.sum()
            idx = np.random.choice(len(self.craters), p=weights)
        else:
            idx = random.randint(0, len(self.craters) - 1)
        return self.craters[idx]

    def __len__(self): return len(self.craters)

# ── Copy-Paste Augmentation ───────────────────────────────────────────────────
def gaussian_feather_mask(mask_patch, radius):
    blurred = cv2.GaussianBlur(mask_patch.astype(np.float32), (2*radius+1, 2*radius+1), sigmaX=radius)
    return np.clip(blurred / (blurred.max() + 1e-6), 0, 1)

def paste_overlaps_existing(target_mask, y1, x1, ch, cw):
    return (target_mask[y1:y1+ch, x1:x1+cw] > 0).any()

def copy_paste_augment(target_img, target_mask, library, n_pastes=PASTES_PER_IMAGE):
    aug_img  = target_img.copy().astype(np.float32)
    aug_mask = target_mask.copy()
    H, W     = aug_img.shape[:2]
    for _ in range(n_pastes):
        crater = library.sample(prefer_small=True)
        if crater is None: break
        c_img, c_mask = crater['img'], crater['mask']
        ch, cw = c_img.shape[:2]
        if ch >= H or cw >= W: continue
        placed = False
        for _ in range(MAX_PASTE_ATTEMPTS):
            y1 = random.randint(0, H - ch); x1 = random.randint(0, W - cw)
            if not paste_overlaps_existing(aug_mask, y1, x1, ch, cw):
                placed = True; break
        if not placed: continue
        weight = gaussian_feather_mask(c_mask, BLEND_FEATHER_RADIUS)
        if aug_img.ndim == 3 and c_img.ndim == 2: c_img = cv2.cvtColor(c_img, cv2.COLOR_GRAY2BGR)
        elif aug_img.ndim == 2 and c_img.ndim == 3: c_img = cv2.cvtColor(c_img, cv2.COLOR_BGR2GRAY)
        if aug_img.ndim == 3:
            w3 = weight[:, :, np.newaxis]
            aug_img[y1:y1+ch, x1:x1+cw] = (w3*c_img.astype(np.float32) + (1-w3)*aug_img[y1:y1+ch, x1:x1+cw])
        else:
            aug_img[y1:y1+ch, x1:x1+cw] = (weight*c_img.astype(np.float32) + (1-weight)*aug_img[y1:y1+ch, x1:x1+cw])
        aug_mask[y1:y1+ch, x1:x1+cw] = np.maximum(aug_mask[y1:y1+ch, x1:x1+cw], c_mask)
    return aug_img.astype(np.uint8), aug_mask

# ── Main Tiling Loop ──────────────────────────────────────────────────────────
def get_base_name(filename): return os.path.splitext(filename)[0]
def find_mask(image_base, mask_files):
    for m in mask_files:
        if image_base in m: return m
    return None

def tile_image_and_mask(image_path, mask_path, base_name, library):
    image = cv2.imread(image_path)
    mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if image is None or mask is None:
        print(f'  [ERROR] Cannot read: {image_path}'); return 0, 0
    h, w = image.shape[:2]; orig_count = aug_count = 0
    for y in range(0, h - TILE_SIZE + 1, STRIDE):
        for x in range(0, w - TILE_SIZE + 1, STRIDE):
            img_patch  = image[y:y+TILE_SIZE, x:x+TILE_SIZE]
            mask_patch = mask[y:y+TILE_SIZE,  x:x+TILE_SIZE]
            if (mask_patch > 0).sum() < 10 or has_black_strip(img_patch): continue
            name = f'{base_name}_x{x}_y{y}.png'
            cv2.imwrite(os.path.join(OUTPUT_IMAGE_DIR, name), img_patch)
            cv2.imwrite(os.path.join(OUTPUT_MASK_DIR,  name), mask_patch)
            orig_count += 1; library.extract_from_tile(img_patch, mask_patch)
            if ENABLE_COPY_PASTE and len(library) > 5:
                for aug_idx in range(AUGMENTED_COPIES):
                    aug_img, aug_mask = copy_paste_augment(img_patch, mask_patch, library)
                    if (aug_mask > 0).sum() < 10: continue
                    aug_name = f'{base_name}_x{x}_y{y}_aug{aug_idx}.png'
                    cv2.imwrite(os.path.join(OUTPUT_IMAGE_DIR, aug_name), aug_img)
                    cv2.imwrite(os.path.join(OUTPUT_MASK_DIR,  aug_name), aug_mask)
                    aug_count += 1
    return orig_count, aug_count

def run_tiling():
    image_files = os.listdir(IMAGE_DIR); mask_files = os.listdir(MASK_DIR)
    library = CraterLibrary(); total_orig = total_aug = skipped = 0
    for img_file in tqdm(image_files, desc='Tiling images'):
        base = get_base_name(img_file); mask_file = find_mask(base, mask_files)
        if mask_file is None: skipped += 1; continue
        orig, aug = tile_image_and_mask(os.path.join(IMAGE_DIR, img_file),
                                        os.path.join(MASK_DIR,  mask_file), base, library)
        total_orig += orig; total_aug += aug
    print(f'\n{"="*45}')
    print(f'  Original tiles : {total_orig:,}')
    print(f'  Augmented tiles: {total_aug:,}')
    print(f'  Total tiles    : {total_orig + total_aug:,}')
    print(f'  Crater library : {len(library):,} instances')
    print(f'{"="*45}')

# ▶ Run tiling — comment out if tiles already exist
run_tiling()

In [ ]:
class CraterDataset(Dataset):
    """Reads pre-tiled images and binary rim masks."""
    def __init__(self, image_dir, mask_dir, augment=False):
        self.image_dir   = Path(image_dir)
        self.mask_dir    = Path(mask_dir)
        self.augment     = augment
        self.image_paths = sorted(self.image_dir.glob('*.png'))
        assert len(self.image_paths) > 0, f'No tiles found in {image_dir}'

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        img_path  = self.image_paths[idx]
        mask_path = self.mask_dir / img_path.name
        img  = cv2.imread(str(img_path),  cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        mask = (cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)
        if self.augment:
            img, mask = self._augment(img, mask)
        return torch.from_numpy(img).unsqueeze(0), torch.from_numpy(mask).unsqueeze(0)

    def _augment(self, img, mask):
        if random.random() > 0.5: img, mask = np.fliplr(img).copy(), np.fliplr(mask).copy()
        if random.random() > 0.5: img, mask = np.flipud(img).copy(), np.flipud(mask).copy()
        if random.random() > 0.5:
            k = random.choice([1, 2, 3])
            img, mask = np.rot90(img, k).copy(), np.rot90(mask, k).copy()
        if random.random() > 0.6:
            img = np.clip(img * random.uniform(0.8, 1.2), 0, 1)
        if random.random() > 0.7:
            img = np.clip(img + np.random.normal(0, 0.02, img.shape).astype(np.float32), 0, 1)
        return img, mask


def make_dataloaders(cfg):
    full_ds = CraterDataset(cfg.tile_image_dir, cfg.tile_mask_dir)
    n       = len(full_ds)
    n_train = int(n * cfg.train_ratio)
    n_val   = int(n * cfg.val_ratio)
    n_test  = n - n_train - n_val
    torch.manual_seed(cfg.seed)
    train_ds, val_ds, test_ds = random_split(full_ds, [n_train, n_val, n_test])

    train_aug    = CraterDataset(cfg.tile_image_dir, cfg.tile_mask_dir, augment=True)
    train_subset = Subset(train_aug, train_ds.indices)

    # persistent_workers=True keeps worker processes alive across epochs — faster data loading
    pw = cfg.num_workers > 0

    train_loader = DataLoader(train_subset, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=cfg.pin_memory,
                              drop_last=True, persistent_workers=pw)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size * 2, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=cfg.pin_memory,
                              persistent_workers=pw)
    test_loader  = DataLoader(test_ds, batch_size=cfg.batch_size * 2, shuffle=False,
                              num_workers=cfg.num_workers, persistent_workers=pw)

    print(f'  Train: {len(train_subset):,}  |  Val: {len(val_ds):,}  |  Test: {len(test_ds):,}')
    return train_loader, val_loader, test_loader, test_ds.indices


print('Dataset classes defined')

In [ ]:
# ── Preview a few tiles ───────────────────────────────────────────────────────
def preview_tiles(n=4):
    img_dir  = Path(CFG.tile_image_dir)
    mask_dir = Path(CFG.tile_mask_dir)
    paths = sorted(img_dir.glob('*.png'))[:n]
    if not paths:
        print('No tiles found — run tiling first.'); return
    fig, axes = plt.subplots(2, n, figsize=(4*n, 8))
    fig.suptitle('Sample Tiles — Image (top) | Mask (bottom)', fontsize=13, fontweight='bold')
    for i, p in enumerate(paths):
        img  = cv2.imread(str(p),              cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(mask_dir / p.name), cv2.IMREAD_GRAYSCALE)
        axes[0, i].imshow(img,  cmap='gray'); axes[0, i].axis('off')
        axes[0, i].set_title(p.name[:20], fontsize=7)
        axes[1, i].imshow(mask, cmap='gray'); axes[1, i].axis('off')
    plt.tight_layout(); plt.show()

preview_tiles()

# **Model Architectures**

In [ ]:
# ── Shared Building Blocks ────────────────────────────────────────────────────
class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(ConvBnRelu(in_ch, out_ch), ConvBnRelu(out_ch, out_ch))
    def forward(self, x): return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConv(in_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
        return self.conv(torch.cat([skip, x], dim=1))

print('Shared blocks defined')

In [ ]:
# ── 1. Baseline U-Net ─────────────────────────────────────────────────────────
class UNet(nn.Module):
    """Standard U-Net — performance baseline."""
    def __init__(self, in_channels=1, out_channels=1, features=(64, 128, 256, 512)):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pools    = nn.ModuleList()
        self.decoders = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.encoders.append(DoubleConv(ch, f))
            self.pools.append(nn.MaxPool2d(2)); ch = f
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)
        rev = list(reversed(features)); in_ch = features[-1] * 2
        for f in rev:
            self.decoders.append(UpBlock(in_ch, f, f)); in_ch = f
        self.head = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x); skips.append(x); x = pool(x)
        x = self.bottleneck(x)
        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)
        return self.head(x)

print('UNet defined')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

# ── Dense Components ──────────────────────────────────────────────────────────
class DenseLayer(nn.Module):
    def __init__(self, in_ch, growth_rate, dropout=0.1):
        super().__init__()
        self.norm = nn.BatchNorm2d(in_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv = nn.Conv2d(in_ch, growth_rate, 3, padding=1, bias=False)
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        out = self.conv(self.relu(self.norm(x)))
        out = self.dropout(out)
        return torch.cat([x, out], dim=1)

class DenseBlock(nn.Module):
    def __init__(self, in_ch, growth_rate=16, num_layers=4, dropout=0.1):
        super().__init__()
        layers = []
        ch = in_ch
        for _ in range(num_layers):
            layers.append(DenseLayer(ch, growth_rate, dropout))
            ch += growth_rate

        self.block = nn.Sequential(*layers)
        self.out_channels = ch

    def forward(self, x):
        # ✅ Gradient checkpointing (critical for memory)
        return checkpoint(self.block, x)

class TransitionDown(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm2d(in_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        return self.block(x)

class TransitionUp(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 1, bias=False)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

# ── DenseUNet ────────────────────────────────────────────────────────────────
class DenseUNet(nn.Module):
    """
    Balanced DenseUNet:
    - Memory safe on Kaggle
    """
    def __init__(self, in_channels=1, out_channels=1,
                 init_features=40, growth_rate=16, num_layers=4):
        super().__init__()

        self.init_conv = ConvBnRelu(in_channels, init_features)

        # ── Encoder ──
        self.enc_blocks = nn.ModuleList()
        self.tds = nn.ModuleList()
        self.skip_channels = []

        ch = init_features
        for _ in range(4):
            db = DenseBlock(ch, growth_rate, num_layers)
            self.enc_blocks.append(db)

            self.skip_channels.append(db.out_channels)

            td = TransitionDown(db.out_channels, db.out_channels // 2)
            self.tds.append(td)

            ch = db.out_channels // 2

        # ── Bottleneck (controlled size) ──
        self.bottleneck = DenseBlock(ch, growth_rate, num_layers)
        ch = self.bottleneck.out_channels

        # ── Decoder ──
        self.tus = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()

        for skip_ch in reversed(self.skip_channels):
            self.tus.append(TransitionUp(ch + skip_ch, skip_ch))

            db = DenseBlock(skip_ch, growth_rate, num_layers)
            self.dec_blocks.append(db)

            ch = db.out_channels

        self.head = nn.Conv2d(ch, out_channels, 1)

    def forward(self, x):
        x = self.init_conv(x)

        skips = []
        for enc, td in zip(self.enc_blocks, self.tds):
            x = enc(x)
            skips.append(x)
            x = td(x)

        x = self.bottleneck(x)

        for tu, dec, skip in zip(self.tus, self.dec_blocks, reversed(skips)):
            x = tu(x, skip)
            x = dec(x)

        return self.head(x)


print("DenseUNet ready")

In [ ]:
# ── 3. TransUNet ──────────────────────────────────────────────────────────────
class PatchEmbedding(nn.Module):
    def __init__(self, in_ch, d_model, patch_size=1):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, d_model, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        x = self.proj(x); B, D, h, w = x.shape
        return x.flatten(2).transpose(1, 2), h, w

class TransformerBlock(nn.Module):
    def __init__(self, d_model=768, num_heads=12, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        mlp_dim = int(d_model * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, mlp_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_dim, d_model), nn.Dropout(dropout))
    def forward(self, x):
        xn = self.norm1(x); attn_out, _ = self.attn(xn, xn, xn)
        x = x + attn_out
        return x + self.ffn(self.norm2(x))

class TransUNet(nn.Module):
    """TransUNet — ResNet-50 CNN encoder + Vision Transformer."""
    def __init__(self, in_channels=1, out_channels=1, img_size=512,
                 d_model=768, num_heads=12, num_layers=12, dropout=0.1):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.DEFAULT)
        backbone.conv1 = nn.Conv2d(in_channels, 64, 7, stride=2, padding=3, bias=False)
        self.enc0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.enc1 = backbone.layer1; self.enc2 = backbone.layer2; self.enc3 = backbone.layer3
        self.patch_embed = PatchEmbedding(1024, d_model, patch_size=1)
        n_patches = (img_size // 16) ** 2
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.transformer = nn.Sequential(
            *[TransformerBlock(d_model, num_heads, dropout=dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.proj_back = nn.Conv2d(d_model, 512, 1)
        self.dec3 = UpBlock(512, 1024, 256); self.dec2 = UpBlock(256, 512, 128)
        self.dec1 = UpBlock(128, 256, 64);   self.dec0 = UpBlock(64, 64, 32)
        self.head = nn.Sequential(
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, out_channels, 1))

    def forward(self, x):
        B = x.shape[0]
        s0 = self.enc0(x); s1 = self.enc1(self.pool(s0))
        s2 = self.enc2(s1); s3 = self.enc3(s2)
        tokens, h, w = self.patch_embed(s3)
        pos = self.pos_embed
        if tokens.shape[1] != pos.shape[1]:
            pos = F.interpolate(
                pos.reshape(1, int(pos.shape[1]**0.5), int(pos.shape[1]**0.5), -1).permute(0,3,1,2),
                size=(h, w), mode='bilinear', align_corners=True
            ).permute(0, 2, 3, 1).reshape(1, h*w, -1)
        tokens = self.transformer(tokens + pos); tokens = self.norm(tokens)
        feat   = tokens.transpose(1, 2).reshape(B, -1, h, w)
        feat   = self.proj_back(feat)
        x = self.dec3(feat, s3); x = self.dec2(x, s2)
        x = self.dec1(x, s1);   x = self.dec0(x, s0)
        return self.head(x)

print('TransUNet defined')

In [ ]:
# ── 4. Attention U-Net (Proposed Enhancement) ─────────────────────────────────
class ChannelAttention(nn.Module):
    """SE-Net style channel attention."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool2d(1); self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(channels, mid, bias=False),
                                nn.ReLU(inplace=True), nn.Linear(mid, channels, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        scale = self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))
        return x * scale.unsqueeze(-1).unsqueeze(-1)

class SpatialAttention(nn.Module):
    """Spatial attention focusing on crater-shaped patterns."""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        desc = torch.cat([x.mean(dim=1, keepdim=True), x.max(dim=1, keepdim=True).values], dim=1)
        return x * self.sigmoid(self.conv(desc))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction); self.sa = SpatialAttention()
    def forward(self, x): return self.sa(self.ca(x))

class AttentionUpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.attn = CBAM(skip_ch)
        self.conv = DoubleConv(in_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x    = self.up(x)
        x    = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
        skip = self.attn(skip)
        return self.conv(torch.cat([skip, x], dim=1))

class AttentionUNet(nn.Module):
    """U-Net + CBAM attention on every skip connection (proposed enhancement)."""
    def __init__(self, in_channels=1, out_channels=1, features=(64, 128, 256, 512)):
        super().__init__()
        self.encoders = nn.ModuleList(); self.pools = nn.ModuleList(); self.decoders = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.encoders.append(DoubleConv(ch, f)); self.pools.append(nn.MaxPool2d(2)); ch = f
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)
        rev = list(reversed(features)); in_ch = features[-1] * 2
        for f in rev:
            self.decoders.append(AttentionUpBlock(in_ch, f, f)); in_ch = f
        self.head = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x); skips.append(x); x = pool(x)
        x = self.bottleneck(x)
        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)
        return self.head(x)

print('AttentionUNet defined')

In [ ]:
# ── 5. RDT-UNet — Residual Dense Transformer UNet (NEW) ──────────────────────
#
#  Encoder  : 4-level Conv blocks with Dense + Residual connections
#    Dense  : each level receives ALL previous levels concatenated + 1×1 compress
#    Residual: shortcut from compress output → after the two 3×3 convs
#
#  Skip connections (attention-filtered before decoder):
#    L1, L2 → CBAM         (channel + spatial, no transformer — maps too large)
#    L3     → Transformer × 1 layer
#    L4     → Transformer × 2 layers  (bottleneck, most semantic)
#
#  Decoder  : 4-level RDTUpBlocks with residual (mirrors encoder)
#  Head     : ConvBnRelu → 1×1 Conv
# ─────────────────────────────────────────────────────────────────────────────

# ── 5a. Encoder block ─────────────────────────────────────────────────────────
class ResidualDenseBlock(nn.Module):
    """
    [dense-concat input] → 1×1 compress → 3×3 conv → 3×3 conv
                                 └──────────────────────┘ (residual add)
    """
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.compress = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        self.conv1 = ConvBnRelu(out_ch, out_ch)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x   = self.compress(x)
        out = self.conv2(self.conv1(x))
        return F.relu(out + x, inplace=True)


# ── 5b. CBAM skip (levels 1 & 2) ─────────────────────────────────────────────
class CBAMSkip(nn.Module):
    """Cheap channel + spatial attention for large feature maps."""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.avg_pool    = nn.AdaptiveAvgPool2d(1)
        self.max_pool    = nn.AdaptiveMaxPool2d(1)
        self.channel_mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
        )
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm2d(1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C = x.shape[:2]
        ca = torch.sigmoid(
            self.channel_mlp(self.avg_pool(x)) + self.channel_mlp(self.max_pool(x))
        ).view(B, C, 1, 1)
        x     = x * ca
        avg_s = x.mean(dim=1, keepdim=True)
        max_s = x.max(dim=1, keepdim=True).values
        sa    = torch.sigmoid(self.spatial_conv(torch.cat([avg_s, max_s], dim=1)))
        return x * sa


# ── 5c. Transformer skip (levels 3 & 4) ──────────────────────────────────────
class TransformerSkip(nn.Module):
    """
    (H×W) positions as tokens → n_layers of self-attention → residual add.
    Pre-LN (norm_first=True) for stable training on small datasets.
    FFN = 2× channels (not 4×) to halve memory vs vanilla transformer.
    """
    def __init__(self, channels: int, n_heads: int = 8, n_layers: int = 2):
        super().__init__()
        while channels % n_heads != 0 and n_heads > 1:
            n_heads -= 1
        layer = nn.TransformerEncoderLayer(
            d_model=channels, nhead=n_heads,
            dim_feedforward=channels * 2,
            dropout=0.1, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.out_norm    = nn.LayerNorm(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        tokens  = x.flatten(2).permute(0, 2, 1)           # (B, H*W, C)
        tokens  = self.out_norm(self.transformer(tokens))  # (B, H*W, C)
        refined = tokens.permute(0, 2, 1).reshape(B, C, H, W)
        return x + refined


# ── 5d. Decoder block ─────────────────────────────────────────────────────────
class RDTUpBlock(nn.Module):
    """Upsample → concat skip → compress → residual conv pair."""
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
        self.fuse = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
        self.conv1 = ConvBnRelu(out_ch, out_ch)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
        x   = self.fuse(torch.cat([x, skip], dim=1))
        out = self.conv2(self.conv1(x))
        return F.relu(out + x, inplace=True)


# ── 5e. Full model ────────────────────────────────────────────────────────────
class RDTUNet(nn.Module):
    """
    Residual Dense Transformer UNet.
    Channels per level: [64, 128, 256, 512]
    """
    def __init__(self, in_channels: int = 1, out_channels: int = 1,
                 base_ch: int = 64, attn_heads: int = 8):
        super().__init__()
        ch = [base_ch * (2 ** i) for i in range(4)]  # [64, 128, 256, 512]

        self.stem = nn.Sequential(ConvBnRelu(in_channels, ch[0]),
                                  ConvBnRelu(ch[0], ch[0]))
        self.pool = nn.MaxPool2d(2, 2)

        # Encoder
        self.enc1 = ResidualDenseBlock(ch[0], ch[0])
        self.enc2 = ResidualDenseBlock(ch[0], ch[1])
        self.enc3 = ResidualDenseBlock(ch[1] + ch[0], ch[2])          # dense L2+L1
        self.enc4 = ResidualDenseBlock(ch[2] + ch[1] + ch[0], ch[3]) # dense L3+L2+L1

        # Skip attention
        self.skip1 = CBAMSkip(ch[0])
        self.skip2 = CBAMSkip(ch[1])
        self.skip3 = TransformerSkip(ch[2], n_heads=attn_heads, n_layers=1)
        self.skip4 = TransformerSkip(ch[3], n_heads=attn_heads, n_layers=2)

        # Decoder
        self.dec3 = RDTUpBlock(ch[3], ch[2], ch[2])
        self.dec2 = RDTUpBlock(ch[2], ch[1], ch[1])
        self.dec1 = RDTUpBlock(ch[1], ch[0], ch[0])

        # Head
        self.head = nn.Sequential(
            ConvBnRelu(ch[0], ch[0] // 2),
            nn.Conv2d(ch[0] // 2, out_channels, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Encoder
        s  = self.stem(x)                                      # (B,  64, H,   W)
        e1 = self.enc1(s)                                      # (B,  64, H,   W)

        p1 = self.pool(e1)                                     # (B,  64, H/2, W/2)
        e2 = self.enc2(p1)                                     # (B, 128, H/2, W/2)

        p2   = self.pool(e2)                                   # (B, 128, H/4, W/4)
        p1x2 = self.pool(p1)                                   # (B,  64, H/4, W/4)
        e3   = self.enc3(torch.cat([p2, p1x2], dim=1))        # (B, 256, H/4, W/4)

        p3   = self.pool(e3)                                   # (B, 256, H/8, W/8)
        p2x2 = self.pool(p2)                                   # (B, 128, H/8, W/8)
        p1x3 = self.pool(p1x2)                                # (B,  64, H/8, W/8)
        e4   = self.enc4(torch.cat([p3, p2x2, p1x3], dim=1)) # (B, 512, H/8, W/8)

        # Attention-filtered skips
        sk1 = self.skip1(e1)   # CBAM  (B,  64, H,   W)
        sk2 = self.skip2(e2)   # CBAM  (B, 128, H/2, W/2)
        sk3 = self.skip3(e3)   # Tr×1  (B, 256, H/4, W/4)
        sk4 = self.skip4(e4)   # Tr×2  (B, 512, H/8, W/8)

        # Decoder
        d3 = self.dec3(sk4, sk3)   # (B, 256, H/4, W/4)
        d2 = self.dec2(d3,  sk2)   # (B, 128, H/2, W/2)
        d1 = self.dec1(d2,  sk1)   # (B,  64, H,   W)

        return self.head(d1)        # (B,   1, H,   W)


print('RDT-UNet defined')

In [ ]:
# ── Model Factory (rdt_unet added) ──────────────────────────────────────────
MODELS = {
    'unet'          : UNet,
    'dense_unet'    : DenseUNet,
    'transunet'     : TransUNet,
    'attention_unet': AttentionUNet,
    'rdt_unet'      : RDTUNet,          # ← NEW
}

def build_model(name, **kwargs):
    if name not in MODELS:
        raise ValueError(f'Unknown model: {name}. Choose from: {list(MODELS.keys())}')
    model = MODELS[name](**kwargs)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  [{name}]  Parameters: {n:,}')
    return model

# ── Sanity check (CPU only) ───────────────────────────────────────────────────
print(f'\nSanity-checking: {MODEL_NAME} (on CPU)...')
x_test = torch.randn(1, CFG.in_channels, 512, 512)
try:
    m = build_model(MODEL_NAME, in_channels=CFG.in_channels, out_channels=CFG.out_channels)
    with torch.no_grad():
        out = m(x_test)
    print(f'  ✓ {MODEL_NAME:20s}  Output shape: {tuple(out.shape)}')
    del m
except Exception as e:
    print(f'  ✗ {MODEL_NAME}  ERROR: {e}')
    raise

# **Loss Functions**

In [ ]:
class DiceLoss(nn.Module):
    """Soft Dice Loss — optimises overlap directly (ideal for sparse rim masks)."""
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, logits, targets):
        p = torch.sigmoid(logits).view(-1); t = targets.view(-1)
        inter = (p * t).sum()
        return 1.0 - (2.0 * inter + self.smooth) / (p.sum() + t.sum() + self.smooth)

class CombinedLoss(nn.Module):
    """Weighted BCE + Dice. BCE pos_weight handles extreme class imbalance."""
    def __init__(self, bce_weight=0.5, dice_weight=0.5, pos_weight=10.0):
        super().__init__()
        self.bw = bce_weight; self.dw = dice_weight
        self.bce  = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))
        self.dice = DiceLoss()
    def forward(self, logits, targets):
        return self.bw * self.bce(logits, targets) + self.dw * self.dice(logits, targets)

class SegmentationMetrics:
    """Accumulates TP/FP/FN and computes IoU, Dice, Precision, Recall, F1."""
    def __init__(self, threshold=0.5):
        self.threshold = threshold; self.reset()
    def reset(self):
        self.tp = self.fp = self.fn = self.tn = 0
    def update(self, logits, targets):
        preds   = (torch.sigmoid(logits) > self.threshold).float().view(-1).cpu()
        targets = targets.view(-1).cpu()
        self.tp += ((preds == 1) & (targets == 1)).sum().item()
        self.fp += ((preds == 1) & (targets == 0)).sum().item()
        self.fn += ((preds == 0) & (targets == 1)).sum().item()
        self.tn += ((preds == 0) & (targets == 0)).sum().item()
    def compute(self):
        tp, fp, fn, eps = self.tp, self.fp, self.fn, 1e-7
        iou  = tp / (tp + fp + fn + eps)
        dice = (2 * tp) / (2 * tp + fp + fn + eps)
        prec = tp / (tp + fp + eps)
        rec  = tp / (tp + fn + eps)
        f1   = (2 * prec * rec) / (prec + rec + eps)
        return {k: round(v, 4) for k, v in
                zip(['iou', 'dice', 'precision', 'recall', 'f1'], [iou, dice, prec, rec, f1])}

print('Loss functions and metrics defined')

In [ ]:
def gpu_mem_str() -> str:
    if torch.cuda.is_available():
        used  = torch.cuda.memory_allocated() / 1024**2
        total = torch.cuda.get_device_properties(0).total_memory / 1024**2
        return f"{used:.0f}/{total:.0f} MB"
    return "CPU"

def delta_str(current: float, previous) -> str:
    if previous is None: return ""
    diff = current - previous
    return f"({'↑' if diff >= 0 else '↓'}{abs(diff):.4f})"

def format_time(seconds: float) -> str:
    m, s = divmod(int(seconds), 60); return f"{m:02d}:{s:02d}"

def print_sep(char="─", w=70): print(char * w)
def print_header(title, w=70):
    print_sep("═", w); pad = (w - len(title) - 2) // 2
    print("║" + " "*pad + title + " "*(w-pad-len(title)-2) + "║")
    print_sep("═", w)


class Trainer:
    """
    Single-model trainer with:
      • AMP (mixed-precision) for GPU speed-up
      • Gradient clipping
      • CosineAnnealingLR scheduler
      • Rich per-batch and per-epoch logging
      • Best-checkpoint saving + early stopping
    """

    def __init__(self, model, model_name: str, cfg,
                 train_loader, val_loader, log_every_n_batches: int = 10):
        self.model        = model.to(DEVICE)
        self.model_name   = model_name
        self.cfg          = cfg
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.log_every    = log_every_n_batches

        self.criterion = CombinedLoss(cfg.bce_weight, cfg.dice_weight, cfg.pos_weight).to(DEVICE)
        self.optimizer = AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=cfg.epochs, eta_min=1e-6)

        # AMP scaler — no-op on CPU
        self.scaler = GradScaler(enabled=USE_AMP)

        self.metrics = SegmentationMetrics()
        self.run_dir = Path(cfg.output_dir) / model_name
        self.run_dir.mkdir(parents=True, exist_ok=True)

        self.best_dice = 0.0; self.best_epoch = 0; self.patience = 0
        self.history   = {k: [] for k in
                          ['train_loss', 'val_loss', 'val_iou', 'val_dice',
                           'val_precision', 'val_recall', 'lr']}
        self._prev_val_dice = self._prev_val_iou = self._prev_val_loss = None

    # ── Train one epoch ───────────────────────────────────────────────────────
    def train_epoch(self, epoch: int) -> float:
        self.model.train()
        total_loss = 0.0; n_batches = len(self.train_loader)
        epoch_start = time.time()

        for batch_idx, (imgs, masks) in enumerate(self.train_loader, 1):
            imgs  = imgs.to(DEVICE, non_blocking=True)   # non_blocking speeds up GPU transfer
            masks = masks.to(DEVICE, non_blocking=True)

            self.optimizer.zero_grad(set_to_none=True)   # set_to_none is faster than zero_grad()

            with autocast(enabled=USE_AMP):               # fp16 forward pass
                logits = self.model(imgs)
                loss   = self.criterion(logits, masks)

            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()

            total_loss += loss.item()

            if batch_idx % self.log_every == 0 or batch_idx == n_batches:
                elapsed   = time.time() - epoch_start
                eta_batch = elapsed / batch_idx * (n_batches - batch_idx)
                print(f"    Batch [{batch_idx:>4d}/{n_batches}] {batch_idx/n_batches*100:5.1f}% | "
                      f"Loss: {loss.item():.4f} (avg: {total_loss/batch_idx:.4f}) | "
                      f"GradNorm: {grad_norm:.3f} | ETA: {format_time(eta_batch)} | GPU: {gpu_mem_str()}")

        return total_loss / n_batches

    # ── Validate one epoch ────────────────────────────────────────────────────
    @torch.no_grad()
    def val_epoch(self) -> tuple:
        self.model.eval(); self.metrics.reset(); total_loss = 0.0
        print("    Validating", end="", flush=True)
        for i, (imgs, masks) in enumerate(self.val_loader):
            imgs  = imgs.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            with autocast(enabled=USE_AMP):
                logits = self.model(imgs)
                total_loss += self.criterion(logits, masks).item()
            self.metrics.update(logits, masks)
            if i % 5 == 0: print(".", end="", flush=True)
        print(" done")
        return total_loss / len(self.val_loader), self.metrics.compute()

    # ── Full fit loop ─────────────────────────────────────────────────────────
    def fit(self) -> Dict:
        n_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print_header(f" {self.model_name.upper()} ")
        print(f"  Parameters   : {n_params:,}")
        print(f"  Device       : {DEVICE}  (AMP={USE_AMP})")
        print(f"  Epochs       : {self.cfg.epochs}")
        print(f"  Batch size   : {self.cfg.batch_size}")
        print(f"  LR           : {self.cfg.learning_rate}  (cosine annealing → 1e-6)")
        print(f"  Loss         : BCE({self.cfg.bce_weight}) + Dice({self.cfg.dice_weight}), pos_weight={self.cfg.pos_weight}")
        print(f"  Early stop   : patience={self.cfg.early_stop_patience}")
        print(f"  Train batches: {len(self.train_loader)}  |  Val batches: {len(self.val_loader)}")
        print_sep()

        total_start = time.time(); epoch_times = []

        for epoch in range(1, self.cfg.epochs + 1):
            epoch_start = time.time()
            lr_now = self.optimizer.param_groups[0]['lr']

            print(f"\n  ┌─ Epoch {epoch:03d}/{self.cfg.epochs} {'─'*30} LR: {lr_now:.2e} ─┐")
            print(f"  │  [TRAIN]")
            train_loss = self.train_epoch(epoch)
            print(f"  │  [VALID]")
            val_loss, scores = self.val_epoch()
            self.scheduler.step()

            epoch_time = time.time() - epoch_start
            epoch_times.append(epoch_time)
            eta_total  = np.mean(epoch_times) * (self.cfg.epochs - epoch)

            for k, v in [('train_loss', train_loss), ('val_loss', val_loss),
                         ('val_iou', scores['iou']), ('val_dice', scores['dice']),
                         ('val_precision', scores['precision']), ('val_recall', scores['recall']),
                         ('lr', lr_now)]:
                self.history[k].append(v)

            dice_d = delta_str(scores['dice'], self._prev_val_dice)
            iou_d  = delta_str(scores['iou'],  self._prev_val_iou)
            loss_d = delta_str(val_loss,        self._prev_val_loss)

            print(f"  │")
            print(f"  │  ┌──────────────── EPOCH {epoch:03d} RESULTS ────────────────┐")
            print(f"  │  │  Train Loss : {train_loss:.6f}")
            print(f"  │  │  Val   Loss : {val_loss:.6f}   {loss_d}")
            print(f"  │  │  {'─'*45}")
            print(f"  │  │  IoU        : {scores['iou']:.6f}   {iou_d}")
            print(f"  │  │  Dice       : {scores['dice']:.6f}   {dice_d}")
            print(f"  │  │  Precision  : {scores['precision']:.6f}")
            print(f"  │  │  Recall     : {scores['recall']:.6f}")
            print(f"  │  │  F1         : {scores['f1']:.6f}")
            print(f"  │  │  {'─'*45}")
            print(f"  │  │  Epoch time : {format_time(epoch_time)}  |  ETA: {format_time(eta_total)}")
            print(f"  │  │  GPU Memory : {gpu_mem_str()}")
            print(f"  │  └─────────────────────────────────────────────────┘")

            self._prev_val_dice = scores['dice']
            self._prev_val_iou  = scores['iou']
            self._prev_val_loss = val_loss

            if scores['dice'] > self.best_dice:
                imp = scores['dice'] - self.best_dice
                self.best_dice = scores['dice']; self.best_epoch = epoch; self.patience = 0
                torch.save({'epoch': epoch, 'model_state': self.model.state_dict(),
                            'optimizer': self.optimizer.state_dict(), 'scaler': self.scaler.state_dict(),
                            'val_dice': self.best_dice, 'val_scores': scores, 'history': self.history},
                           self.run_dir / 'best_model.pth')
                print(f"  │  NEW BEST! Dice +{imp:.4f} → {self.best_dice:.4f}  [checkpoint saved]")
            else:
                self.patience += 1
                print(f"  │  ⏳ No improvement. Patience: {self.patience}/{self.cfg.early_stop_patience}")

            print(f"  └{'─'*58}┘")

            if self.patience >= self.cfg.early_stop_patience:
                print(f"\n  ⛔ Early stopping at epoch {epoch} (best epoch {self.best_epoch}, Dice={self.best_dice:.4f})")
                break

        total_time = time.time() - total_start
        self._print_summary(total_time, epoch)
        with open(self.run_dir / 'history.json', 'w') as f:
            json.dump(self.history, f, indent=2)

        return {'model': self.model_name, 'best_val_dice': self.best_dice,
                'best_epoch': self.best_epoch, 'history': self.history,
                'time_min': round(total_time / 60, 2)}

    def _print_summary(self, total_time, last_epoch):
        print_sep("═")
        print(f"  TRAINING COMPLETE — {self.model_name.upper()}")
        print_sep("═")
        print(f"  Total time    : {format_time(total_time)} ({total_time/60:.1f} min)")
        print(f"  Epochs run    : {last_epoch}  |  Best epoch: {self.best_epoch}")
        print(f"  Best Val Dice : {self.best_dice:.6f}")
        if self.history['val_dice']:
            print(f"  Final Val IoU : {self.history['val_iou'][-1]:.6f}")
        chars = "▁▂▃▄▅▆▇█"
        for key, label in [('val_loss', 'Val Loss'), ('val_dice', 'Val Dice')]:
            vals = self.history[key]
            if len(vals) > 1:
                mn, mx = min(vals), max(vals) + 1e-9; rng = mx - mn
                spark = "".join(chars[min(int((v-mn)/rng*(len(chars)-1)), len(chars)-1)] for v in vals[-40:])
                print(f"  {label} trend: {spark}")
        print_sep("═")
        print(f"  Checkpoint → {self.run_dir / 'best_model.pth'}")
        print(f"  History    → {self.run_dir / 'history.json'}")
        print_sep("═")

print("Trainer defined (AMP-enabled)")

# **Train Dense-UNet**

In [ ]:
# ── Build shared dataloaders ──────────────────────────────────────────────────
print(f'Loading dataset for model: {MODEL_NAME}...')
train_loader, val_loader, test_loader, test_indices = make_dataloaders(CFG)

In [ ]:
# Reinstall PyTorch matching Kaggle's CUDA environment
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118 -q
import torch
print("PyTorch version:", torch.__version__)
print("CUDA version PyTorch built with:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU compute capability:", torch.cuda.get_device_capability(0))

In [ ]:
# ── Optional: torch.compile for extra speed (PyTorch >= 2.0 only) ─────────────
# Uncomment the compile line if you're on PyTorch 2.x and want ~10-30% more speed.
# Note: compile adds ~1-2 min of warmup on the first epoch.
USE_COMPILE = False   # Set True to enable torch.compile

set_seed(CFG.seed)

print(f'\n{"#"*55}')
print(f'  TRAINING: {MODEL_NAME.upper()}')
print(f'{"#"*55}')

model = build_model(MODEL_NAME, in_channels=CFG.in_channels, out_channels=CFG.out_channels).to(DEVICE)

if USE_COMPILE and hasattr(torch, 'compile'):
    print('  Compiling model with torch.compile()...')
    model = torch.compile(model)
    print(' Model compiled')

trainer = Trainer(model, MODEL_NAME, CFG, train_loader, val_loader)
result  = trainer.fit()

# **Evaluation**

In [ ]:
@torch.no_grad()
def evaluate_test(model, model_name, test_loader, run_dir):
    model.eval(); metrics = SegmentationMetrics()
    for imgs, masks in tqdm(test_loader, desc=f'  Test [{model_name}]', leave=False):
        imgs  = imgs.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
        metrics.update(logits, masks)
    scores = metrics.compute(); scores['model'] = model_name
    out_path = Path(run_dir) / model_name / 'test_scores.json'
    with open(out_path, 'w') as f:
        json.dump(scores, f, indent=2)
    print(f'  Test scores saved → {out_path}')
    return scores

# ── Load best checkpoint, then evaluate ──────────────────────────────────────
ckpt_path = Path(CFG.output_dir) / MODEL_NAME / 'best_model.pth'
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    # Strip _orig_mod prefix if model was compiled
    state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state'].items()}
    model.load_state_dict(state)
    print(f'  Loaded best checkpoint (epoch {ckpt["epoch"]}, val_dice={ckpt["val_dice"]:.4f})')

test_scores = evaluate_test(model, MODEL_NAME, test_loader, CFG.output_dir)

print(f'\n{"="*50}')
print(f'  TEST RESULTS — {MODEL_NAME.upper()}')
print(f'{"="*50}')
for k, v in test_scores.items():
    if k != 'model':
        print(f'  {k:12s}: {v}')
print(f'{"="*50}')

In [ ]:
# ── Training Curves ───────────────────────────────────────────────────────────
def plot_training_curves(model_name=MODEL_NAME):
    hist_path = Path(CFG.output_dir) / model_name / 'history.json'
    if not hist_path.exists():
        print(f'No history found for {model_name}'); return
    with open(hist_path) as f:
        hist = json.load(f)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Training Curves — {model_name.upper()}', fontsize=14, fontweight='bold')
    ep = range(1, len(hist['train_loss']) + 1)

    axes[0].plot(ep, hist['train_loss'], '--', label='Train Loss', alpha=0.7)
    axes[0].plot(ep, hist['val_loss'],         label='Val Loss',   lw=2)
    axes[0].set_title('Loss (BCE + Dice)'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(ep, hist['val_dice'],      label='Dice', lw=2)
    axes[1].plot(ep, hist['val_iou'],       label='IoU',  lw=2)
    axes[1].set_title('Validation Metrics'); axes[1].set_xlabel('Epoch')
    axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(ep, hist['lr'], color='orange', lw=2)
    axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    out = Path(CFG.output_dir) / model_name / 'training_curves.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    print(f'Saved → {out}')
    plt.show()

plot_training_curves()

In [ ]:
# ── Qualitative Grid: Input | GT | Prediction | Error Overlay ─────────────────
@torch.no_grad()
def predict(model, img_np):
    model.eval()
    t = torch.from_numpy(img_np.astype(np.float32)/255.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
    with autocast(enabled=USE_AMP):
        out = torch.sigmoid(model(t))
    return (out.squeeze().cpu().float().numpy() > 0.5).astype(np.uint8) * 255

def overlay_masks(img, gt, pred):
    out = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR).astype(np.float32)
    gt_b, pred_b = gt > 127, pred > 127
    out[ gt_b &  pred_b] = out[ gt_b &  pred_b]*0.4 + np.array([0,  200, 0  ])*0.6  # TP green
    out[ gt_b & ~pred_b] = out[ gt_b & ~pred_b]*0.4 + np.array([0,  0,   200])*0.6  # FN blue
    out[~gt_b &  pred_b] = out[~gt_b &  pred_b]*0.4 + np.array([200, 0,  0  ])*0.6  # FP red
    return out.astype(np.uint8)

def plot_qualitative(model_name=MODEL_NAME, n_samples=5):
    ckpt_path = Path(CFG.output_dir) / model_name / 'best_model.pth'
    m = build_model(model_name, in_channels=1, out_channels=1).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state'].items()}
    m.load_state_dict(state)

    img_dir  = Path(CFG.tile_image_dir); mask_dir = Path(CFG.tile_mask_dir)
    all_pngs = sorted(img_dir.glob('*.png'))
    if not all_pngs:
        print('No tiles found.'); return
    samples = random.sample(all_pngs, min(n_samples, len(all_pngs)))

    fig, axes = plt.subplots(n_samples, 4, figsize=(16, 4*n_samples))
    if n_samples == 1: axes = axes[np.newaxis, :]
    fig.suptitle(f'Qualitative Results — {model_name.upper()}', fontsize=15, fontweight='bold')
    for ax, t in zip(axes[0], ['Input Image', 'Ground Truth', 'Prediction', 'Error Overlay']):
        ax.set_title(t, fontsize=12, fontweight='bold')

    for i, p in enumerate(samples):
        img  = cv2.imread(str(p),               cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(mask_dir/p.name), cv2.IMREAD_GRAYSCALE)
        pred    = predict(m, img)
        overlay = overlay_masks(img, mask, pred)
        axes[i,0].imshow(img,  cmap='gray'); axes[i,0].axis('off')
        axes[i,1].imshow(mask, cmap='gray'); axes[i,1].axis('off')
        axes[i,2].imshow(pred, cmap='gray'); axes[i,2].axis('off')
        axes[i,3].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); axes[i,3].axis('off')

    handles = [mpatches.Patch(color=(0, 0.78, 0),    label='True Positive'),
               mpatches.Patch(color=(0, 0,    0.78), label='False Negative (missed)'),
               mpatches.Patch(color=(0.78, 0, 0),    label='False Positive')]
    fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.01))
    plt.tight_layout()
    out = Path(CFG.output_dir) / model_name / 'qualitative_grid.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    print(f'Saved → {out}')
    plt.show()

plot_qualitative()

In [ ]:
# ── Crater Size Analysis: Small / Medium / Large ──────────────────────────────
@torch.no_grad()
def analyze_by_size(model_name=MODEL_NAME, n_tiles=200):
    ckpt_path = Path(CFG.output_dir) / model_name / 'best_model.pth'
    m = build_model(model_name, in_channels=1, out_channels=1).to(DEVICE)
    state = {k.replace('_orig_mod.', ''): v
             for k, v in torch.load(ckpt_path, map_location=DEVICE)['model_state'].items()}
    m.load_state_dict(state)

    img_dir  = Path(CFG.tile_image_dir); mask_dir = Path(CFG.tile_mask_dir)
    all_pngs = sorted(img_dir.glob('*.png'))
    paths    = random.sample(all_pngs, min(n_tiles, len(all_pngs)))
    bins     = {'Small\n(<200px)': [], 'Medium\n(200-2k)': [], 'Large\n(>2000px)': []}

    for p in tqdm(paths, desc=f'Size analysis [{model_name}]'):
        img  = cv2.imread(str(p),               cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(mask_dir/p.name), cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None: continue
        pred = predict(m, img)
        _, labels, stats, _ = cv2.connectedComponentsWithStats(
            (mask > 127).astype(np.uint8), connectivity=8)
        for i in range(1, len(stats)):
            area = stats[i, cv2.CC_STAT_AREA]; comp_mask = (labels == i).astype(np.uint8)
            comp_pred = ((pred > 127).astype(np.uint8)) * comp_mask
            tp = int((comp_mask & comp_pred).sum())
            fp = int((~comp_mask.astype(bool) & comp_pred.astype(bool)).sum())
            fn = int((comp_mask.astype(bool) & ~comp_pred.astype(bool)).sum())
            iou = tp / (tp + fp + fn + 1e-7)
            key = ('Small\n(<200px)' if area < 200 else
                   'Medium\n(200-2k)' if area < 2000 else 'Large\n(>2000px)')
            bins[key].append(iou)

    fig, ax = plt.subplots(figsize=(8, 5))
    cats   = list(bins.keys())
    means  = [np.mean(v) if v else 0 for v in bins.values()]
    counts = [len(v) for v in bins.values()]
    bars   = ax.bar(cats, means, color=['#2196F3', '#4CAF50', '#FF5722'],
                    alpha=0.85, edgecolor='black', width=0.5)
    for bar, m_val, c in zip(bars, means, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'IoU={m_val:.3f}\n(n={c})', ha='center', va='bottom', fontsize=11)
    ax.set_title(f'IoU by Crater Size — {model_name.upper()}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Mean IoU'); ax.set_ylim(0, 1.15); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    out = Path(CFG.output_dir) / model_name / 'size_analysis.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    print(f'Saved → {out}')
    plt.show()
    return {k.split('\n')[0]: round(float(np.mean(v)), 4) if v else 0 for k, v in bins.items()}

size_results = analyze_by_size()
print('Size-wise IoU:', size_results)

In [ ]:
# ── Final summary for this model ──────────────────────────────────────────────
print('\n' + '='*55)
print(f'  RESULTS SUMMARY — {MODEL_NAME.upper()}')
print('='*55)

score_path = Path(CFG.output_dir) / MODEL_NAME / 'test_scores.json'
if score_path.exists():
    with open(score_path) as f:
        sc = json.load(f)
    for k, v in sc.items():
        if k != 'model':
            print(f'  {k:12s}: {v}')
else:
    print('  Run Section 9 first to generate test scores.')

print(f'\n  Checkpoint : {Path(CFG.output_dir) / MODEL_NAME / "best_model.pth"}')
print(f'  History    : {Path(CFG.output_dir) / MODEL_NAME / "history.json"}')
print('='*55)